In [0]:
dbutils.fs.mkdirs("/Volumes/bronze/default/raw/iot_data/")

In [0]:
display(dbutils.fs.ls("/Volumes/bronze/default/raw/iot_data//"))

In [0]:
data2 = """{"device_id":104,"temperature":22.8,"city":"Pune","timestamp":"2026-03-06 10:03:00"}
{"device_id":105,"temperature":26.2,"city":"Mumbai","timestamp":"2026-03-06 10:04:00"}"""

dbutils.fs.put("/Volumes/bronze/default/raw/iot_data/data2.json", data2, True)

In [0]:
raw_path = "/Volumes/bronze/default/raw/iot_data/"
schema_path = "/Volumes/bronze/default/raw/iot_schema/"
checkpoint_path = "/Volumes/bronze/default/raw/iot_checkpoint/"

df_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", schema_path)
         .option("cloudFiles.inferColumnTypes", "true")
         .load(raw_path)
)

query = (
    df_stream.writeStream
             .format("delta")
             .outputMode("append")
             .option("checkpointLocation", checkpoint_path)
             .trigger(availableNow=True)
             .toTable("bronze")
)

query.awaitTermination()

In [0]:
%sql
SELECT * FROM bronze

In [0]:
data3 = """{"device_id":106,"temperature":21.9,"city":"Pune","timestamp":"2026-03-06 10:05:00"}
{"device_id":107,"temperature":27.3,"city":"Bangalore","timestamp":"2026-03-06 10:06:00"}"""

dbutils.fs.put("/Volumes/bronze/default/raw/iot_data/data3.json", data3, True)